# M03 – Exceptions (PCAP 2.1–2.2)

**PCAP Alignment**: Section 2 (2.1–2.2). **Professional Focus**: Defensive design, custom exception hierarchy.

---
## Learning Outcomes

- Use **except** variants (except E as e, (E1,E2), try/except/else); understand **exception hierarchy**.

- Use **raise** and re-raise; use **assert** for invariants.

- Define and use **custom exception** classes; use **e.args**.

---
## Table of Contents

1. except Variants and Hierarchy (PCAP 2.1)
2. raise and assert (PCAP 2.1)
3. Custom Exceptions (PCAP 2.2)
4. Built-in and Edge Cases
5. More Examples
6. Practice

In [ ]:
try:
    int("x")
except ValueError as e:
    print("Caught:", e.args)
class MyError(Exception): pass
try:
    raise MyError("test")
except MyError as e:
    print("Custom exception caught:", e)

---
## 1. except Variants (PCAP 2.1)

**Catching exceptions**

- **try:** / **except ExceptionType as e:** — Run the code under **try**; if an exception of type **ExceptionType** (or a subclass) is raised, run the **except** block and bind the exception instance to **e**. You can then inspect **e.args**, **str(e)**, or custom attributes.

- **except (E1, E2):** — Catch **either** **E1** or **E2** (or their subclasses) in one handler. Useful when you want the same handling for several types.

- **try / except / else:** — The **else** block runs only when the **try** block **did not** raise an exception. Use it for code that should run on success only (e.g. parsing succeeded, so use the parsed value).

**Exception hierarchy**

- **BaseException** is the top of the tree. **Exception** is the usual base for "normal" exceptions (e.g. **ValueError**, **TypeError**, **ZeroDivisionError**). **KeyboardInterrupt** and **SystemExit** inherit from **BaseException** but not **Exception**. Always list **except** clauses from **most specific** to **least specific** (e.g. **except ValueError** before **except Exception**), because the first matching handler runs and the rest are skipped.

In [ ]:
try:
    int("x")
except ValueError as e:
    print("Caught:", e.args)
try:
    1 / 0
except (ValueError, ZeroDivisionError) as e:
    print("Caught:", type(e).__name__, e.args)

In [ ]:
# Exception hierarchy: ValueError and ZeroDivisionError are subclasses of Exception
print("ValueError subclass of Exception:", issubclass(ValueError, Exception))
print("ZeroDivisionError subclass of Exception:", issubclass(ZeroDivisionError, Exception))
# KeyboardInterrupt is NOT a subclass of Exception (it's under BaseException)
print("KeyboardInterrupt subclass of Exception:", issubclass(KeyboardInterrupt, Exception))

In [ ]:
# else block: runs only when no exception occurred
try:
    value = int("100")
except ValueError:
    print("Invalid number")
else:
    print("Parsed successfully, value =", value)

---
## 2. raise and assert (PCAP 2.1)

**Raising exceptions**

- **raise SomeError("msg")** — Creates an instance of **SomeError** with the given message (or other arguments, stored in **e.args**) and raises it. Execution jumps to the nearest matching **except** handler. Use built-in types (**ValueError**, **TypeError**) or your own exception classes.

- **raise** (with no expression) — Used **inside** an **except** block to **re-raise** the same exception that was just caught. Useful when you log or handle partially but want the exception to propagate to the caller.

**assert**

- **assert condition, "msg"** — If **condition** is false, raises **AssertionError** with the optional message. Intended for **invariants** and developer checks (e.g. "this list is not empty"). **Do not** use **assert** for validating user input or external data — it can be disabled with **python -O**. Use **if not valid: raise ValueError(...)** instead.

In [ ]:
try:
    raise ValueError("custom message")
except ValueError as e:
    print("e.args:", e.args)
# assert 2 + 2 == 4
# assert False, "optional message"  # would raise AssertionError

In [ ]:
# assert: raises AssertionError when condition is False
# assert 2 + 2 == 5, "Math is broken"  # uncomment to see AssertionError
# Better for user input: use if/raise instead of assert (assert can be disabled with python -O)
def validate_positive(n):
    if n <= 0:
        raise ValueError("n must be positive")
    return n
print("validate_positive(5):", validate_positive(5))

In [ ]:
# Re-raise: catch, log or handle, then propagate the same exception
try:
    try:
        int("not a number")
    except ValueError as e:
        print("Caught ValueError, re-raising:", e.args)
        raise  # re-raise the same exception
except ValueError as e2:
    print("Outer handler received:", type(e2).__name__, e2.args)

---
## 3. Custom Exceptions (PCAP 2.2)

You can define your own exception types by **subclassing** **Exception** (or a built-in like **ValueError**). This lets callers catch **your** exception specifically and keeps error handling clear.

- **Define:** e.g. **class ValidationError(Exception): pass**. You can add an **__init__** to store extra data (e.g. a field name or error code); the arguments are available in **e.args**.

- **Raise:** **raise ValidationError("invalid input")** or **raise ValidationError("field X", code=42)** if you define a custom **__init__**.

- **Catch:** **except ValidationError as e:** — only this type (and subclasses) are caught. You can catch a **base** exception (e.g. **Exception**) to handle all user-defined and many built-in errors in one place, or catch the specific type for finer control.

- **Use case:** Domain-specific errors (e.g. **ValidationError**, **NotFoundError**, **AuthError**) make the intent clear and allow callers to react differently to different failures.

---
## 4. Built-in and Edge Cases

- **e.args**: tuple of arguments passed to the exception (e.g. **raise ValueError("msg")** → **e.args == ("msg",)**).

- **Order of except**: Put **more specific** exceptions first; the first matching handler runs. **except Exception** before **except ValueError** would catch everything and ValueError would never run.

- **Bare except:** Avoid **except:**; it catches KeyboardInterrupt/SystemExit. Prefer **except Exception** or specific types.

- **assert**: Can be disabled with **python -O**. Do not use for user input validation; use **if/raise**.

In [ ]:
# e.args: access arguments passed to the exception
try:
    raise ValueError("first arg", "second arg")
except ValueError as e:
    print("e.args:", e.args)
    print("str(e):", str(e))

In [ ]:
# Custom exception with an extra attribute (optional)
class ConfigError(Exception):
    def __init__(self, message, key=None):
        super().__init__(message)
        self.key = key

try:
    raise ConfigError("Missing required key", key="api_key")
except ConfigError as e:
    print("Message:", e.args[0])
    print("Key:", e.key)

---
## 5. More Examples

**Example: try/except/else – run code only when no exception**

In [ ]:
try:
    x = int("42")
except ValueError:
    print("Invalid number")
else:
    print("OK, value is", x)

---
## 6. Practice

**Practice 1:** Catch **ValueError** when calling **int("abc")** and print **e.args**.

In [ ]:
# Order matters: specific exception first. This catches ValueError in the first handler.
try:
    int("x")
except ValueError as e:
    print("ValueError caught first:", e.args)
except Exception as e:
    print("Exception (would not run for ValueError):", e)

In [ ]:
try:
    int("abc")
except ValueError as e:
    print("e.args:", e.args)

**Practice 2:** Define **class ValidationError(Exception): pass**. Raise **ValidationError("invalid input")** and catch it; print the message.

In [ ]:
class ValidationError(Exception):
    pass
try:
    raise ValidationError("invalid input")
except ValidationError as e:
    print(e.args[0])

**Practice 3:** Use **assert** to check that a variable **n** is positive; trigger **AssertionError** with **n = -1**.

In [ ]:
n = -1
# assert n > 0, "n must be positive"  # uncomment to see AssertionError
if n <= 0:
    raise ValueError("n must be positive")  # better for user input